# LDM(Latent Diffusion Model)

前面我们尝试 DDPM 的模型都是设置图像在 64 的大小的，因为设置得太大的话，演示生成会非常的慢，相信大家也有所体会。我们已经简单学习了 VAE，它就是把图像先压缩到一个 latent 表示，再还原回原始图像。那么可不可以把 DPPM 的过程放到 **latent space 进行训练**呢？本篇的 **Latent Diffusion Model(LDM)** 就实现了这么一个过程。

<div align="center">
    <img src="./imgs/LDM_architecture.png" alt="LDM_architecture" style="width: 600px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/582693939)

[DLM 论文](https://arxiv.org/pdf/2112.10752)

[LDM Pytorch 实现](https://github.com/explainingai-code/StableDiffusion-PyTorch)

## LDM 核心思想简述

我们先从**大体上**看看作者的思路

### 图片压缩感知

这一步其实就是使用 VAE 把图像空间的图像压缩成潜在空间的感知，比如把原始的[3, 128, 128]的图像压缩成潜在空间的[4, 16, 16]的感知，并使用这个感知来对 DDPM 进行训练的一个过程，这就是 LDM 训练的第一阶段；

作者使用了两类 VAE 进行实验，分别叫做 **KL-reg** 和 **VQ-reg**，这里我们再**简单回顾**一下，具体细节请移步对应篇章进行了解:

- KL-reg:

    其实就是标准的 VAE，编码器不直接输出潜在表示$z$，而是输出一个均值和方差，并通过 KL 散度约束这个均值方差构造的分布接近标准高斯分布，从而能够直接从标准高斯分布中采样$z$，再进行解码

- VQ-reg:

    VQVAE，编码器还是直接输出的潜在表示$z_e$，但是会有一个 codebook 存储一些向量，并通过最近邻替换$z_e$里的每一个向量为 codebook 里的向量得到$z_q$，再进行解码；采样的一个例子就是使用 PixelCNN，自回归采样出所有位置需要的 codebook 的向量对应的位置索引，再取出向量组装得到 $z_q$，解码得到采样输出


VAE 训练的损失函数我以 **KL-reg** 为例进行讲解:

VAE 的损失函数由**重建损失、KL 散度损失和感知损失**组成:

$$
\mathcal{L} = \mathcal{L}_{\text{recon}} + 
\mathcal{L}_{\text{kl}} \cdot \text{KL\_weight} + 
\mathcal{L}_{\text{perceptual}}
$$

其中:
- $\mathcal{L}_{\text{recon}}$: 就是原始图像和重建图像的均方误差

- $\mathcal{L}_{\text{kl}}$: 用于约束编码器输出构造的高斯分布尽可能符合标准高斯分布$\mathcal{N}(0, 1)$

- $\mathcal{L}_{\text{perceptual}}$: 使用一个预训练模型(比如VGG)对原始图像和重建图像进行处理得到两个特征表示，计算这两个特征表示的均方误差衡量它们的**结构相似度**

### LDM 过程

其实就是在潜在空间中完成 DDPM 过程得到$z$，再使用 VAE 对采样的$z$进行解码的过程，LDM 学习的目标函数如下:

$$
\mathcal{L} = \mathbb{E}_{z, \epsilon \sim \mathcal{N}(0, 1), t}
\left[ \parallel \epsilon - \epsilon_\theta(z_t, t) \parallel_2^2 \right]
$$

其中$z$是 VAE 编码图像$x$得到的潜在空间表示，训练完成后，就可以按照上面的说法采样得到新的图像

### 条件机制

目的就是得到一个条件去噪模型$\epsilon_\theta(z_t, t, y)$，其中$y$为条件，通过我们前面讲述的条件控制 **CFG** 进行条件生成，但是需要把条件$y$嵌入到模型当中进行训练；

作者为了适应多种模态的条件，就引入了领域专用编码器$\tau_\theta(y)$对条件进行编码，从而能够让图像嵌入到模型的中间层的表示中，原论文使用**交叉注意力**把条件融入到中间表示中，其中$Q$来自中间表示$z$，$K$、$V$来自条件编码$\tau_\theta(y)$

### 源码实现

下面我们将使用简单的 VAE 压缩图像，并在 latent space 中训练 DDPM(with CFG)，并使用训练完成的模型进行采样(如果你理解了 VAE 和 DDPM with CFG，其实很容易就能实现)

### VAE

在这里补充一下，实际实现时，会在训练的后期阶段引入对抗学习，也就是使用一个判别器进行对抗学习，那么 VAE 的损失就多了一个**对抗损失**，即需要让判别器尽可能判断生成的图像为 True，而判别器就需要尽可能判断生成的图像为 False，原始图像为 True，并与 VAE 一同进行参数更新

**请注意**，你需要下载一个[权重文件(vgg.pth)](https://github.com/richzhang/PerceptualSimilarity/tree/master/lpips/weights/v0.1)，并放在当期目录的 weights/v0.1/vgg.pth 位置

In [ ]:
from c_ldm import (
    get_config, get_vae, get_ddpm_cfg,
    LPIPS, Discriminator,
    train_vae, train_ldm
)
import torch
import warnings
warnings.filterwarnings("ignore")


config = get_config("./configs/vae_ldm_config.yaml")
train_vae_config = config["train_vae"]
vae = get_vae(config)

# 100 epochs 训练示例，实际10000 epochs，可以转跳到对应函数查看实现
train_vae(
    vae=vae,
    img_size=config["global"]["img_size"],
    checkpoint_save_dir=train_vae_config["checkpoint_save_dir"],
    batch_size=train_vae_config["batch_size"],
    lr=train_vae_config["lr"],
    epochs=train_vae_config["epochs"],
    dataset_dir=train_vae_config["dataset_dir"],
    disc_weight=train_vae_config["disc_weight"],
    perceptual_weight=train_vae_config["perceptual_weight"],
    kl_weight=train_vae_config["kl_weight"],
    autoencoder_acc_steps=train_vae_config["autoencoder_acc_steps"],
    disc_start=train_vae_config["disc_start"]
)


Loading model from: d:\my_github_file\Casual_Paper_Learning\LDM\weights\v0.1\vgg.pth


Training VAE: 100%|██████████| 100/100 [00:28<00:00,  3.46it/s, recon=0.195, kl=3.19, perceptual=0.589, disc=0.2, gen=0.141] 

[trainer]Saved: vae_epoch_100.pth


[trainer]Saved: vae_iter100.pth


In [ ]:
config = get_config("./configs/vae_ldm_config.yaml")
vae = get_vae(config, load_pretrained=True)

vae.recon_test(dataset_dir="./train", save_dir="./recon/vae", batch_size=2, img_size=128)


可以看看重建效果:

<div align="center">
    <img src="./recon/vae/compare_batch0_0.png" alt="compare_batch0_0" style="width: 400px; height: auto;">
</div>

<div align="center">
    <img src="./recon/vae/compare_batch0_1.png" alt="compare_batch0_1" style="width: 400px; height: auto;">
</div>


### DDPM(with CFG)

其实就是使用训练好的 VAE 把图像压缩到隐空间中，在隐空间中完成 DDPM 训练过程就可以了，这里我不使用交叉注意力，采用简单的相加融入(参考 DDPM CFG 我的实现)

In [2]:
config = get_config("./configs/vae_ldm_config.yaml")
train_cfg = config["train_ddpm_cfg"]

vae = get_vae(config, load_pretrained=True)
ddpm_cfg = get_ddpm_cfg(config)

# 100 epochs 训练示例，实际10000 epochs
train_ldm(
    pretrained_vae=vae,
    ddpm_cfg=ddpm_cfg,
    checkpoint_save_dir=train_cfg["checkpoint_save_dir"],
    batch_size=train_cfg["batch_size"],
    lr=train_cfg["lr"],
    epochs=train_cfg["epochs"],
    dataset_dir=train_cfg["dataset_dir"],
    num_classes=train_cfg["num_classes"],
    img_size=train_cfg["img_size"]
)


[VAE]Loaded checkpoint from ckpts/vae/vae_iter10000.pth


Training DDPM_CFG: 100%|██████████| 100/100 [00:25<00:00,  3.93it/s, avg_loss=0.635]


[trainer]Saved: cfg_ddpm_unet_iter100.pth


### 采样实现

采样过程也很好理解，就是在隐空间中使用 DDPM 采样完得到$z$之后，再送入 VAE 解码器，就能得到最终的采样图像了

In [3]:
from c_ldm import CFGDenoiseDiffusion, VAE 
from torchvision.utils import save_image
from typing import Optional
import os
import torch


def sample(ddpm_cfg:CFGDenoiseDiffusion, vae:VAE, num_sample=4, save_dir="./sample", 
           class_label:Optional[int]=None, z_img_size=16, z_channels=4):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ddpm_cfg.eps_model.to(device)
    ddpm_cfg.eps_model.eval()
    vae.to(device)
    vae.eval()

    with torch.no_grad():
        if class_label is not None:
            z = ddpm_cfg.sample_with_cfg(c=class_label, sample_num=num_sample, img_size=z_img_size, 
                                         img_channels=z_channels)
        else:
            z = ddpm_cfg.sample(sample_num=num_sample, img_size=z_img_size, img_channels=z_channels)

        z = z / 0.18215
        imgs = vae.decode(z)

        imgs = torch.clamp(imgs, -1., 1.)
        imgs = (imgs + 1) / 2

    for i in range(num_sample):
        img_name = f"sample_class{class_label}_{i}.png" if class_label is not None else f"sample_random_{i}.png"
        save_image(imgs[i], os.path.join(save_dir, img_name))

    print(f"[Sampler]Saved {num_sample} samples to {save_dir}")
    

In [4]:
config = get_config("./configs/vae_ldm_config.yaml")
vae = get_vae(config, load_pretrained=True)
ddpm_cfg = get_ddpm_cfg(config, load_pretrained=True)

sample(ddpm_cfg=ddpm_cfg, vae=vae, num_sample=4, save_dir="./sample", class_label=None, 
        z_img_size=16, z_channels=4)  # random
sample(ddpm_cfg=ddpm_cfg, vae=vae, num_sample=4, save_dir="./sample", class_label=0, 
       z_img_size=16, z_channels=4)  # class 0
sample(ddpm_cfg=ddpm_cfg, vae=vae, num_sample=4, save_dir="./sample", class_label=1, 
       z_img_size=16, z_channels=4)  # class 1


[VAE]Loaded checkpoint from ckpts/vae/vae_iter10000.pth
[DDPM_CFG]Loaded checkpoint from ckpts/ddpm_cfg/cfg_ddpm_unet_iter10000.pth


Sampling without CFG: 100%|██████████| 1000/1000 [00:29<00:00, 34.18it/s]


[Sampler]Saved 4 samples to ./sample


Sampling with CFG, class: 0: 100%|██████████| 1000/1000 [00:57<00:00, 17.27it/s]


[Sampler]Saved 4 samples to ./sample


Sampling with CFG, class: 1: 100%|██████████| 1000/1000 [00:57<00:00, 17.31it/s]


[Sampler]Saved 4 samples to ./sample


可以看看 sample 的效果:

<h5 align="center">random</h5> 

<div align="center">
    <img src="./sample/sample_random_0.png" style="width: 150px; height: auto;">
    <img src="./sample/sample_random_1.png" style="width: 150px; height: auto;">
    <img src="./sample/sample_random_2.png" style="width: 150px; height: auto;">
    <img src="./sample/sample_random_3.png" style="width: 150px; height: auto;">
</div>

<h5 align="center">class 0</h5> 

<div align="center">
    <img src="./sample/sample_class0_0.png" style="width: 150px; height: auto;">
    <img src="./sample/sample_class0_1.png" style="width: 150px; height: auto;">
    <img src="./sample/sample_class0_2.png" style="width: 150px; height: auto;">
    <img src="./sample/sample_class0_3.png" style="width: 150px; height: auto;">
</div>

<h5 align="center">class 1</h5> 

<div align="center">
    <img src="./sample/sample_class1_0.png" style="width: 150px; height: auto;">
    <img src="./sample/sample_class1_1.png" style="width: 150px; height: auto;">
    <img src="./sample/sample_class1_2.png" style="width: 150px; height: auto;">
    <img src="./sample/sample_class1_3.png" style="width: 150px; height: auto;">
</div>